In [1]:
!pip install gradio ultralytics

  Using cached gradio-6.13.0-py3-none-any.whl.metadata (17 kB)
  Using cached fastapi-0.136.1-py3-none-any.whl.metadata (28 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached huggingface_hub-1.12.0-py3-none-any.whl.metadata (14 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.26-py3-none-any.whl.metadata (2.1 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached starlette-1.0.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached tomlkit-0.14.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached typer-0.25.0-py3-none-any.whl.metadata (15 kB)
  Using cached uvicorn-0.46.0-py3-none-any.whl.metadata (

In [1]:
import gradio as gr
import torch
import numpy as np
import cv2
from PIL import Image
from ultralytics import YOLO
import os

# ===============================
# DEVICE SETUP
# ===============================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

# ===============================
# CLASS LABELS
# ===============================

CLASS_NAMES = [
    "Aortic enlargement","Atelectasis","Calcification","Cardiomegaly",
    "Consolidation","ILD","Infiltration","Lung Opacity",
    "Nodule-Mass","Other lesion","Pleural effusion","Pleural thickening",
    "Pneumothorax","Pulmonary fibrosis"
]

# ===============================
# LOAD YOLOv12 MODEL
# ===============================

MODEL_PATH = r"D:\VS code\ChestXrayProject\cxray14\runs\detect\runs\cxr14_yolov12m\weights\best.pt"

print("Loading model from:", MODEL_PATH)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError("Model file not found")

model = YOLO(MODEL_PATH)
model.to(DEVICE)

print("Model loaded successfully")

# ===============================
# IOU FUNCTION
# ===============================

def calculate_iou(boxA, boxB):

    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB-xA) * max(0, yB-yA)

    areaA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])

    return inter / (areaA + areaB - inter + 1e-6)

# ===============================
# DETECTION FUNCTION
# ===============================

def detect_xray(image, label_file):

    img = np.array(image.convert("RGB"))
    h, w = img.shape[:2]

    results = model.predict(img, conf=0.4, device=DEVICE, verbose=False)

    boxes = results[0].boxes.xyxy.cpu().numpy()
    classes = results[0].boxes.cls.cpu().numpy()
    scores = results[0].boxes.conf.cpu().numpy()

    gt_boxes = []
    gt_classes = []

    if label_file is not None:

        with open(label_file.name) as f:
            lines = f.read().splitlines()

        for l in lines:

            c, cx, cy, bw, bh = map(float, l.split())

            x1 = (cx-bw/2)*w
            y1 = (cy-bh/2)*h
            x2 = (cx+bw/2)*w
            y2 = (cy+bh/2)*h

            gt_boxes.append([x1,y1,x2,y2])
            gt_classes.append(int(c))

    gt_diseases = [CLASS_NAMES[c] for c in gt_classes]
    predicted_diseases = []

    img_draw = img.copy()

    # ===============================
    # DRAW GROUND TRUTH
    # ===============================

    for b,c in zip(gt_boxes,gt_classes):

        x1,y1,x2,y2 = map(int,b)

        cv2.rectangle(img_draw,(x1,y1),(x2,y2),(0,255,0),2)

        cv2.putText(
            img_draw,
            CLASS_NAMES[c],
            (x1,y1-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,255,0),
            2
        )

    output_text = []

    # ===============================
    # DRAW PREDICTIONS
    # ===============================

    for b,c,s in zip(boxes,classes,scores):

        x1,y1,x2,y2 = map(int,b)

        disease = CLASS_NAMES[int(c)]
        confidence = float(s)

        predicted_diseases.append(disease)

        label = f"{disease} ({confidence:.2f})"

        cv2.rectangle(img_draw,(x1,y1),(x2,y2),(255,0,0),2)

        cv2.putText(
            img_draw,
            label,
            (x1,y2+20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255,0,0),
            2
        )

        best_iou = 0
        best_class = None

        for gt_b,gt_c in zip(gt_boxes,gt_classes):

            iou = calculate_iou(b,gt_b)

            if iou > best_iou:
                best_iou = iou
                best_class = CLASS_NAMES[gt_c]

        if best_class is not None:

            if best_iou > 0.5 and disease == best_class:
                status = "Correct Detection"

            elif best_iou > 0.5 and disease != best_class:
                status = "Wrong Disease (Box Correct)"

            else:
                status = "Poor Localization"

            text_block = (
                f"Disease: {disease}\n"
                f"Confidence: {confidence:.2f}\n"
                f"Ground Truth: {best_class}\n"
                f"IoU: {best_iou:.2f}\n"
                f"Status: {status}"
            )

        else:

            text_block = (
                f"Disease: {disease}\n"
                f"Confidence: {confidence:.2f}\n"
                f"No ground truth label provided"
            )

        output_text.append(text_block)

    # ===============================
    # MISSED DISEASES
    # ===============================

    missed = [d for d in gt_diseases if d not in predicted_diseases]

    # ===============================
    # EXTRA DETECTIONS
    # ===============================

    extra = [d for d in predicted_diseases if d not in gt_diseases]

    if missed:
        output_text.append("\nMissed Diseases:")
        for m in missed:
            output_text.append(f"- {m}")

    if extra:
        output_text.append("\nExtra Detected Diseases:")
        for e in extra:
            output_text.append(f"- {e}")

    if len(output_text) == 0:
        output_text = ["No findings detected"]

    return img, Image.fromarray(img_draw), "\n".join(output_text)

# ===============================
# GRADIO UI
# ===============================

with gr.Blocks(theme=gr.themes.Soft(), title="Chest X-ray Detection") as demo:

    gr.Markdown("# 🫁 Chest X-ray Detection")

    gr.Markdown(
    """
    🟢 **Green Box** → Ground Truth Label  
    🔴 **Red Box** → Model Prediction  
    """
    )

    with gr.Row():

        with gr.Column():

            input_img = gr.Image(type="pil", label="Upload Chest X-ray")

            label_file = gr.File(label="Ground Truth Label (.txt)")

            run_btn = gr.Button("Run Detection")

        with gr.Column():

            original = gr.Image(label="Original X-ray")

            result = gr.Image(label="Detection Result")

    predictions = gr.Textbox(
        lines=20,
        label="Detection Summary"
    )

    run_btn.click(
        detect_xray,
        inputs=[input_img,label_file],
        outputs=[original,result,predictions]
    )

# ===============================
# RUN SERVER
# ===============================

if __name__ == "__main__":
    demo.launch()

c:\Users\maazm\anaconda3\envs\chest_xray\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on: cuda
Loading model from: D:\VS code\ChestXrayProject\cxray14\runs\detect\runs\cxr14_yolov12m\weights\best.pt
Model loaded successfully


C:\Users\maazm\AppData\Local\Temp\ipykernel_13388\1849459721.py:221: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Chest X-ray Detection") as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
